# ChuckleNet Scale221 — Kaggle GPU Training

**Pipeline:**
1. Download embeddings (~57MB) from public URL
2. Pseudo-label with fusion teacher model
3. Train fusion model (5-fold GroupKFold)
4. Save model + results

**Runtime:** ~10-15 min on T4 GPU

In [ ]:
# Cell 1: Setup + Download data from Kaggle dataset
!pip install -q torch scikit-learn tqdm

import os
WORK = '/kaggle/working'
os.makedirs(f'{WORK}/scale221', exist_ok=True)
os.chdir(f'{WORK}/scale221')

# Download from Kaggle dataset subhajitdas/scale221
!kaggle datasets download -d subhajitdas/scale221 --unzip -p .
print('✓ Data downloaded')

# Extract if needed
import zipfile
if os.path.exists('embeddings.zip'):
    print('Extracting...')
    with zipfile.ZipFile('embeddings.zip', 'r') as z:
        z.extractall('.')
    print('✓ Extracted')


In [ ]:
# Cell 2: Extract embeddings + load video IDs
import zipfile, os

if os.path.exists('embeddings.zip'):
    print('Extracting embeddings...')
    with zipfile.ZipFile('embeddings.zip', 'r') as z:
        z.extractall('.')
    print('✓ Extracted')
else:
    print('embeddings.zip not found!')

import json
if os.path.exists('video_ids.json'):
    with open('video_ids.json') as f:
        video_ids = json.load(f)
    print(f'Video IDs: {len(video_ids)}')
else:
    print('video_ids.json not found!')

import numpy as np
emb_files = sorted([f for f in os.listdir('embeddings') if f.endswith('.npy')])
print(f'Embedding files: {len(emb_files)}')

In [ ]:
# Cell 4: Verify files loaded
import json
import numpy as np

# Files should be in /kaggle/working/scale221/
os.chdir('/kaggle/working/scale221')

with open('video_ids.json') as f:
    video_ids = json.load(f)
print(f'Video IDs: {len(video_ids)}')

emb_files = sorted([f for f in os.listdir('embeddings') if f.endswith('.npy')])
print(f'Embedding files: {len(emb_files)}')

X_list, vids_list = [], []
for f in emb_files:
    vid = f.replace('.npy', '')
    emb = np.load(f'embeddings/{f}')
    X_list.append(emb)
    vids_list.extend([vid] * len(emb))

X_all = np.vstack(X_list)
vids_all = vids_list
print(f'Total: {len(vids_all)} segments, {len(set(vids_all))} videos')
print(f'Feature shape: {X_all.shape}')


In [ ]:
# Cell 4: Load teacher + pseudo-label
import torch
import torch.nn as nn
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# Load teacher model
FUSION_MODEL = '/kaggle/input/scale221/best_fusion_model.pt'
if not os.path.exists(FUSION_MODEL):
    FUSION_MODEL = 'best_fusion_model.pt'

teacher = FusionMLP(input_dim=791)
teacher.load_state_dict(torch.load(FUSION_MODEL, map_location='cpu'), strict=False)
teacher.to(device)
teacher.eval()
print('✓ Teacher loaded')

# Pseudo-label in batches
print('Pseudo-labeling...')
BATCH = 2048
probs_list = []
for i in tqdm(range(0, len(X_all), BATCH)):
    batch = torch.tensor(X_all[i:i+BATCH], dtype=torch.float32).to(device)
    with torch.no_grad():
        probs_list.append(teacher(batch).cpu().numpy().squeeze())
probs = np.concatenate(probs_list)

y_new = (probs >= 0.5).astype(int)
pos_rate = y_new.mean()
print(f'Pseudo: {y_new.sum()}/{len(y_new)} pos ({100*pos_rate:.1f}%)')
print(f'Prob: min={probs.min():.4f}, max={probs.max():.4f}, mean={probs.mean():.4f}')

# Fallback: top 30%
if pos_rate < 0.15:
    print('Using top 30% as positive')
    threshold = np.percentile(probs, 70)
    y_new = (probs >= threshold).astype(int)
    print(f'New rate: {100*y_new.mean():.1f}%')

y_all = y_new
print(f'Final: {y_all.sum()}/{len(y_all)} pos ({100*y_all.mean():.1f}%)')

In [ ]:
# Cell 5: Train fusion model (5-fold GroupKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

groups = np.array(vids_all)
n_vids = len(set(vids_all))
print(f'Training: {len(y_all)} segs, {n_vids} videos')

gkf = GroupKFold(n_splits=min(5, n_vids))
models, fold_f1s = [], []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups)):
    print(f'\n=== Fold {fold+1} ===')
    Xtr, Xte = X_all[tr_idx], X_all[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]

    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    model = FusionMLP(input_dim=791).to(device)
    pos_rate_tr = ytr.sum() / max(len(ytr), 1)
    pos_weight = min((1.0 - pos_rate_tr) / (pos_rate_tr + 1e-8), 3.0)
    print(f'pos_rate={pos_rate_tr:.3f}, pos_weight={pos_weight:.2f}')

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    criterion = nn.BCELoss(pos_weight=torch.tensor([pos_weight]).to(device))

    Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1).to(device)

    best_f1, patience, no_imp = 0, 5, 0
    for epoch in range(50):
        model.train()
        for i in range(0, len(Xtr_t), 256):
            bx, by = Xtr_t[i:i+256], ytr_t[i:i+256]
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(Xte_s, dtype=torch.float32).to(device)).cpu().numpy().squeeze()
            f = f1_score(yte, (preds >= 0.5).astype(int), zero_division=0)
            if f > best_f1: best_f1 = f; no_imp = 0
            else: no_imp += 1
            if no_imp >= patience: break

    model.eval()
    with torch.no_grad():
        sample_probs = model(torch.tensor(Xte_s[:100], dtype=torch.float32).to(device)).cpu().numpy().squeeze()
        if sample_probs.std() < 0.01:
            print(f'⚠️ SATURATION WARNING')

        preds = model(torch.tensor(Xte_s, dtype=torch.float32).to(device)).cpu().numpy().squeeze()
        p = precision_score(yte, (preds >= 0.5).astype(int), zero_division=0)
        r = recall_score(yte, (preds >= 0.5).astype(int), zero_division=0)
        f = f1_score(yte, (preds >= 0.5).astype(int), zero_division=0)
        print(f'F1={f:.4f} P={p:.4f} R={r:.4f}')

    models.append(model)
    fold_f1s.append(f)

print(f'\n=== CV F1: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f} ===')

In [ ]:
# Cell 6: Save
import json

best_idx = int(np.argmax(fold_f1s))
torch.save(models[best_idx].state_dict(), f'{WORK}/scale221_fusion_model.pt')
print('Model saved')

results = {
    'n_videos': n_vids,
    'n_segments': int(len(y_all)),
    'positive_rate': float(y_all.mean()),
    'cross_val_f1': float(np.mean(fold_f1s)),
    'cross_val_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s],
    'teacher_max_prob': float(probs.max()),
}
with open(f'{WORK}/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))